# Stage B / NB 11 — NV-Reason-CXR-3B: findings agent and abstention analysis (A4)

Protocol reference: agent **A4**; experiments **E0c** (already run in NB 07) and **E5-S**;
research question **RQ2**.

## RQ2 has to be restated, and this notebook is why

NB 07 established two things about this model that invalidate the original RQ2 framing:

1. **Its output space cannot express mRALE.** All 1,262 substantive traces end
   `</think>\n<answer> Atelectasis, Lung Opacity </answer>`; none contain a `{`. It was RL-trained
   to emit a fixed CheXpert-style finding list, so zero-shot mRALE is impossible *by
   construction*, not by prompt design.
2. **It abstains more often the sicker the patient** — 38.5% refusal at mRALE 0 rising to 86.1%
   at 19–24, and the gradient survives conditioning on image brightness.

RQ2 therefore stops being *"does reasoning-tuned pretraining improve severity estimation?"* — a
question this model cannot be asked — and becomes:

> **RQ2 (restated).** Does a reasoning-tuned CXR model contribute usable diagnostic evidence, and
> is its abstention behaviour a surface artefact of output formatting or a deeper limitation?

## What this notebook does

| section | cost | purpose |
| --- | --- | --- |
| 3–4 | free | parse `<answer>` findings from NB 07's cached traces → A4's evidence channel |
| 5 | free | full abstention analysis with confound tests → figure-ready |
| 6 | ~2–3 GPU-h | **answer-format scoring**: score `<answer> Lung Opacity </answer>` against `<answer> No Finding </answer>`, giving A4 a legitimate AUROC in its own output format |
| 7 | not implemented | **E5-S** is reserved for a dedicated future Qwen2.5-VL LoRA notebook |

Sections 3–6 reuse NB 07's cached traces and cost almost nothing. **E5-S is not implemented
here**; enabling its flag raises without writing a false completion marker.

## Why E5-S remains a possible future experiment

The protocol gates E5-S on E0c's zero-shot result, which was poor. But the abstention finding
changes the question. Fine-tuning is the one intervention that *can* alter an output format and an
abstention policy, so E5-S now tests something the paper genuinely wants to know: whether
severity-dependent refusal is a surface behaviour that supervision overrides, or a
representational limit that survives it. Either answer is reportable.

## Outputs (under `stage_B/nb11_nvreason/`)
`nvreason_findings.jsonl`, `abstention_analysis.csv`, `abstention_by_severity.csv`,
`predictions_nvreason.jsonl`, `answer_format_scores.jsonl`, `arm_summary.csv`,
`usability.json`, and `rq2_restatement.json`.

## 1. Imports, seeds, and the Stage A path contract

In [ ]:
import gc
import json
import math
import os
import random
import sys
import time
from collections import Counter, OrderedDict, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# Shared metric definitions. Table 2 is only a valid comparison if every arm uses these.
_METRICS_SEARCH = [Path.cwd(), Path.cwd().parent, Path.cwd() / "stage_B",
                   Path.cwd().parent / "stage_B"]
for _candidate in _METRICS_SEARCH:
    if (_candidate / "cxr_metrics.py").is_file():
        sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError(
        "cxr_metrics.py not found. It must sit beside the Stage B notebooks; every arm in "
        f"Table 2 depends on its metric definitions. Searched: {_METRICS_SEARCH}")
import cxr_metrics as cm

SEED = 42
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

# ---- Stage A path contract -------------------------------------------------------------
FALLBACK_STAGE_A_DIR = Path("/data/liangz2/openi/midrc/tetci_resubmit/stage_A")
PATHS_JSON_CANDIDATES = [
    FALLBACK_STAGE_A_DIR / "nb00_environment" / "stage_a_paths.json",
    Path.cwd() / "stage_a_paths.json",
    Path.cwd().parent / "stage_A" / "nb00_environment" / "stage_a_paths.json",
]
stage_paths = None
for candidate in PATHS_JSON_CANDIDATES:
    if candidate.is_file():
        stage_paths = json.loads(candidate.read_text(encoding="utf-8"))
        print("Path contract:", candidate)
        break
if stage_paths is None:
    raise FileNotFoundError("stage_a_paths.json not found. Run Stage A NB 00 first.")

PROJECT_ROOT = Path(stage_paths["project_root"])
STAGE_ROOT = Path(stage_paths["stage_root"])
STAGE_A_DIR = Path(stage_paths["stage_a_dir"])
STAGE_B_DIR = STAGE_ROOT / "stage_B"
NB01_DIR = Path(stage_paths["nb_output_dirs"]["nb01_inventory"])
NB02_DIR = Path(stage_paths["nb_output_dirs"]["nb02_folds"])
NB03_DIR = Path(stage_paths["nb_output_dirs"]["nb03_external"])
NB04_DIR = Path(stage_paths["nb_output_dirs"]["nb04_localization"])
FOLD_DEF_DIR = NB02_DIR / "fold_definitions"
MODEL_REVISIONS = stage_paths.get("model_revisions", {})

N_FOLDS = 5
INTERNAL_VALIDATION_FRACTION = 0.10   # matches the tested LoRA notebooks

print("Stage B output root:", STAGE_B_DIR)
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| BF16:", torch.cuda.is_bf16_supported())

## 2. Configuration

In [ ]:
import hashlib
import re

NB11_DIR = STAGE_B_DIR / "nb11_nvreason"
NB11_DIR.mkdir(parents=True, exist_ok=True)
NB07_DIR = STAGE_B_DIR / "nb07_zeroshot"

MODEL_ID = "nvidia/NV-Reason-CXR-3B"
MODEL_REVISION = MODEL_REVISIONS.get(MODEL_ID)
AGENT_NAME = "A4_nvreason"
ARM = "E0c_nvreason_findings"

# Its actual output grammar, established from NB 07's traces rather than assumed.
ANSWER_TAG = re.compile(r"<answer>(.*?)</answer>", re.S | re.I)
THINK_TAG = re.compile(r"<think>(.*?)</think>", re.S | re.I)
REFUSAL_PATTERN = re.compile(r"please provide|provide a frontal|cannot see|no image", re.I)
REFUSAL_MAX_CHARS = 200

# Section 6: score in the model's OWN format. Teacher-forcing JSON candidates it never emits
# is what made its NB 07 covid_score meaningless.
RUN_ANSWER_FORMAT_SCORING = True
POSITIVE_ANSWER = "<answer> Lung Opacity </answer>"
NEGATIVE_ANSWER = "<answer> No Finding </answer>"
SCORING_MAX_IMAGES = None          # None = all internal images

# E5-S is reserved but not implemented; enabling it raises in section 8.
RUN_E5S_LORA = False
E5S_FOLDS = [0]                    # one fold is enough to answer the question
E5S_EPOCHS = 3
E5S_LORA_R, E5S_LORA_ALPHA, E5S_LORA_DROPOUT = 32, 64, 0.05
E5S_LEARNING_RATE, E5S_WARMUP = 1e-4, 0.05

print("Model:", MODEL_ID, "| revision:", MODEL_REVISION)
print("NB 07 source:", NB07_DIR)
print("Output:", NB11_DIR)

## 3. Load NB 07's cached traces

Nothing is regenerated. NB 07 already spent the GPU-hours; this notebook reads its artifacts.

In [ ]:
def load_view_index():
    path = NB04_DIR / "view_index.csv"
    if not path.is_file():
        raise FileNotFoundError(f"{path} not found. Run Stage A NB 04 first.")
    frame = pd.read_csv(path)
    print(f"view_index.csv: {len(frame):,} rows, cohorts={dict(Counter(frame['cohort']))}")
    return frame


def load_folds():
    path = FOLD_DEF_DIR / "midrc_folds_v2.csv"
    if not path.is_file():
        raise FileNotFoundError(
            f"{path} not found. Run Stage A NB 02 first. Do NOT fall back to the legacy "
            "multi_task_CV folds: they leak at study level."
        )
    frame = pd.read_csv(path)
    print(f"midrc_folds_v2.csv: {len(frame):,} images, "
          f"{frame['group_id'].nunique():,} groups, folds={dict(sorted(Counter(frame['fold']).items()))}")
    return frame


def build_cohort_table():
    # One row per image: labels + fold + every cached view path. This is the single table
    # every Stage B notebook trains and predicts from.
    views = load_view_index()
    folds = load_folds()

    internal = folds.merge(
        views[views["cohort"] == "MIDRC"].drop(columns=["held_out_fold"], errors="ignore"),
        on="filename", how="inner", suffixes=("", "_view"),
    )
    if len(internal) != len(folds):
        missing = set(folds["filename"]) - set(internal["filename"])
        raise RuntimeError(
            f"{len(missing)} fold images have no NB 04 localization row (e.g. "
            f"{sorted(missing)[:5]}). Re-run NB 04 with MAX_IMAGES=None."
        )
    internal["mrale_right"] = (internal["extent_right_numerical"]
                               * internal["density_right_numerical"])
    internal["mrale_left"] = (internal["extent_left_numerical"]
                              * internal["density_left_numerical"])
    internal["is_external"] = False

    external_rows = []
    external_dir = NB03_DIR / "external_manifests"
    if external_dir.is_dir():
        for manifest_path in sorted(external_dir.glob("*_manifest.csv")):
            frame = pd.read_csv(manifest_path)
            if "status" in frame.columns:
                frame = frame[frame["status"] == "OK"]
            if not len(frame):
                continue
            cohort = str(frame["cohort"].iloc[0])
            merged = frame.merge(
                views[views["cohort"] == cohort][
                    ["filename", "v0_image", "v1_thorax_image", "v2_left_image",
                     "v2_right_image", "left_box", "right_box", "any_fallback"]
                ],
                on="filename", how="inner",
            )
            merged["fold"] = -1
            merged["is_external"] = True
            merged["group_id"] = "external::" + merged["filename"].astype(str)
            for column in ["mrale_total_annotated", "mrale_right", "mrale_left",
                           "extent_right_numerical", "density_right_numerical",
                           "extent_left_numerical", "density_left_numerical"]:
                if column not in merged.columns:
                    merged[column] = np.nan
            if "mrale_total" in merged.columns:
                merged["mrale_total_annotated"] = merged["mrale_total"]
            external_rows.append(merged)

    table = pd.concat([internal] + external_rows, ignore_index=True, sort=False)
    table["image_key"] = table.apply(
        lambda row: f"{row.get('cohort', 'MIDRC')}::{row['filename']}", axis=1)
    print()
    print(f"Cohort table: {len(table):,} rows "
          f"({int((~table['is_external']).sum()):,} internal, "
          f"{int(table['is_external'].sum()):,} external)")
    return table


def grouped_inner_split(subset, fraction, seed):
    # Group-aware inner validation split, same construction as the tested notebooks: whole
    # groups move together so the inner split cannot leak either.
    groups = sorted(subset["group_id"].astype(str).unique())
    rng = random.Random(seed)
    rng.shuffle(groups)
    n_validation = max(1, round(len(groups) * fraction))
    validation_groups = set(groups[:n_validation])
    is_validation = subset["group_id"].astype(str).isin(validation_groups)
    train, validation = subset[~is_validation], subset[is_validation]
    assert not (set(train["group_id"]) & set(validation["group_id"]))
    return train, validation


def ground_truth_fields(row):
    def maybe_int(value):
        return None if value is None or (isinstance(value, float) and math.isnan(value)) else int(value)
    covid = row.get("covid_positive")
    if isinstance(covid, float) and math.isnan(covid):
        covid = None
    return {
        "gt_covid": covid if covid in {"Yes", "No"} else None,
        "gt_mrale_total": maybe_int(row.get("mrale_total_annotated")),
        "gt_mrale_right": maybe_int(row.get("mrale_right")),
        "gt_mrale_left": maybe_int(row.get("mrale_left")),
        "gt_extent_right": maybe_int(row.get("extent_right_numerical")),
        "gt_density_right": maybe_int(row.get("density_right_numerical")),
        "gt_extent_left": maybe_int(row.get("extent_left_numerical")),
        "gt_density_left": maybe_int(row.get("density_left_numerical")),
    }


def evaluate_arm(rows, label):
    # Single entry point for metrics, so every arm in Table 2 is scored identically.
    covid_rows = [row for row in rows if row.get("gt_covid") is not None]
    metrics = {"arm": label, "n_rows": len(rows)}
    if covid_rows:
        metrics["covid"] = cm.classification_metrics(
            [row["gt_covid"] for row in covid_rows],
            [row.get("covid_pred") for row in covid_rows],
            [row.get("covid_score") for row in covid_rows],
        )
    mrale_rows = [row for row in rows if row.get("gt_mrale_total") is not None]
    if mrale_rows:
        metrics["mrale"] = cm.mrale_metrics(mrale_rows)
    metrics["output"] = cm.localization_free_metrics(rows)
    return metrics


def print_arm_summary(metrics):
    covid = metrics.get("covid", {})
    mrale = metrics.get("mrale", {})
    print(f"  {metrics['arm']:<34} "
          f"AUROC={covid.get('auroc', float('nan')):.4f} "
          f"balAcc={covid.get('balanced_accuracy', float('nan')):.4f} "
          f"spec={covid.get('specificity', float('nan')):.4f} | "
          f"mRALE MAE={mrale.get('mae', float('nan')):.3f} "
          f"QWK={mrale.get('qwk', float('nan')):.4f} "
          f"cov={mrale.get('coverage', float('nan')):.3f}")

In [ ]:
trace_path = NB07_DIR / "nvreason_traces.jsonl"
if not trace_path.is_file():
    raise FileNotFoundError(
        f"{trace_path} not found. Run NB 07 first with capture_traces enabled for "
        "E0c_nvreason -- this notebook analyses those traces rather than regenerating them."
    )
traces = cm.read_jsonl(trace_path)
print(f"Traces loaded: {len(traces):,}")

cohort = build_cohort_table()
internal_cohort = cohort[~cohort["is_external"]].reset_index(drop=True)
by_key = {str(row["image_key"]): row for row in internal_cohort.to_dict("records")}
print(f"Internal cohort: {len(internal_cohort):,}")

manifest_path = NB01_DIR / "midrc_manifest.csv"
appearance = {}
if manifest_path.is_file():
    for row in pd.read_csv(manifest_path).to_dict("records"):
        appearance[str(row["filename"])] = {
            "mean_intensity": row.get("mean_intensity"),
            "stddev_intensity": row.get("stddev_intensity"),
            "min_side": min(row.get("width") or 0, row.get("height") or 0),
        }
    print(f"Image-appearance statistics available for {len(appearance):,} images")

## 4. Parse the findings — A4's evidence channel

`<answer>` holds a comma-separated finding list. That, not a severity score, is what this agent
contributes to the reasoner: an independent second opinion on *what is present*, from a model
trained to reason rather than to regress.

In [ ]:
def parse_trace(text):
    text = text or ""
    abstained = bool(REFUSAL_PATTERN.search(text)) and len(text) < REFUSAL_MAX_CHARS
    answer = ANSWER_TAG.search(text)
    think = THINK_TAG.search(text)
    labels = ([item.strip() for item in answer.group(1).split(",") if item.strip()]
              if answer else [])
    return {
        "abstained": abstained,
        "labels": labels,
        "n_labels": len(labels),
        "has_answer_tag": bool(answer),
        "has_think_block": bool(think),
        "reasoning_chars": len(think.group(1)) if think else 0,
        "trace_chars": len(text),
        "contains_json_brace": "{" in text,
    }


findings_rows, parsed_by_key = [], {}
for trace in traces:
    key = str(trace.get("image_key", ""))
    parsed = parse_trace(trace.get("trace"))
    parsed_by_key[key] = parsed
    findings_rows.append({
        "image_key": key, "agent": AGENT_NAME,
        "findings": [{"finding": label, "source": "nvreason_answer_tag"}
                     for label in parsed["labels"]],
        "abstained": parsed["abstained"],
        "n_findings": parsed["n_labels"],
        "reasoning_chars": parsed["reasoning_chars"],
    })
cm.write_jsonl(NB11_DIR / "nvreason_findings.jsonl", findings_rows)

label_counts = Counter(label for row in findings_rows
                       for label in [f["finding"] for f in row["findings"]])
pd.DataFrame(label_counts.most_common(), columns=["finding", "n"]).to_csv(
    NB11_DIR / "nvreason_label_distribution.csv", index=False)

print(f"Findings written for {len(findings_rows):,} images")
print(f"  abstained            : {sum(r['abstained'] for r in findings_rows):,}")
print(f"  with an <answer> tag : {sum(1 for r in findings_rows if r['n_findings']):,}")
print(f"  any JSON brace       : {sum(1 for k, v in parsed_by_key.items() if v['contains_json_brace']):,}")
print()
print("Output vocabulary (this IS the agent's expressive range):")
for label, count in label_counts.most_common(15):
    print(f"  {label:<34} {count:>5}")
print()
print("Note the absence of any numeric severity term. That is the whole of finding 1: mRALE")
print("cannot be requested from this model zero-shot because it is not in its output space.")

## 5. Abstention analysis

The headline result from this arm. Refusal is regressed against severity, and then against
severity **conditioned on image appearance**, because the obvious objection is that whiteout films
are simply bright and flat.

In [ ]:
from scipy.stats import spearmanr

rows = []
for trace in traces:
    key = str(trace.get("image_key", ""))
    parsed = parsed_by_key[key]
    source = by_key.get(key, {})
    filename = key.split("::")[-1]
    total = trace.get("gt_mrale_total")
    rows.append({
        "image_key": key, "abstained": parsed["abstained"],
        "gt_mrale_total": total, "severity_band": cm.severity_band(total),
        "gt_covid": trace.get("gt_covid"),
        "n_labels": parsed["n_labels"], "reasoning_chars": parsed["reasoning_chars"],
        **appearance.get(filename, {}),
    })
abstention = pd.DataFrame(rows)
abstention.to_csv(NB11_DIR / "abstention_analysis.csv", index=False)

band_table = (abstention.dropna(subset=["severity_band"])
              .groupby("severity_band")["abstained"].agg(["sum", "count"]))
band_table["rate"] = (band_table["sum"] / band_table["count"]).round(4)
band_table = band_table.reindex([b for b in ["none", "mild", "moderate", "severe"]
                                 if b in band_table.index])
band_table.to_csv(NB11_DIR / "abstention_by_severity.csv")
print("Abstention by severity band:")
print(band_table.to_string())

scored = abstention.dropna(subset=["gt_mrale_total"])
rho, p_value = spearmanr(scored["abstained"].astype(int), scored["gt_mrale_total"].astype(float))
print()
print(f"Spearman(abstention, mRALE) = {rho:+.4f}  (p = {p_value:.3e}, n = {len(scored):,})")

print()
print("Confound check -- the same gradient within FIXED image-appearance bands:")
confound_rows = []
if "mean_intensity" in abstention.columns and abstention["mean_intensity"].notna().any():
    for low, high in [(80, 120), (120, 160), (160, 200)]:
        window = scored[(scored["mean_intensity"] >= low) & (scored["mean_intensity"] < high)]
        if len(window) < 60:
            continue
        inner = window.groupby("severity_band")["abstained"].agg(["sum", "count"])
        print(f"  mean intensity {low}-{high} (n={len(window):,}):")
        for band in ["none", "mild", "moderate", "severe"]:
            if band in inner.index and int(inner.loc[band, "count"]) >= 15:
                hit, total = int(inner.loc[band, "sum"]), int(inner.loc[band, "count"])
                print(f"      {band:<10} {hit:>4}/{total:<5} = {hit / total:6.1%}")
                confound_rows.append({"intensity_band": f"{low}-{high}", "severity_band": band,
                                      "abstained": hit, "n": total,
                                      "rate": round(hit / total, 4)})
        window_rho = spearmanr(window["abstained"].astype(int),
                               window["gt_mrale_total"].astype(float))[0]
        confound_rows.append({"intensity_band": f"{low}-{high}", "severity_band": "ALL",
                              "abstained": int(window["abstained"].sum()), "n": len(window),
                              "rate": round(float(window["abstained"].mean()), 4),
                              "spearman_within_band": round(float(window_rho), 4)})
    pd.DataFrame(confound_rows).to_csv(NB11_DIR / "abstention_confound.csv", index=False)
    print()
    print("  If the gradient persists within every intensity band, brightness does not explain")
    print("  it and the behaviour is a property of the model, not of the radiographs.")
else:
    print("  Skipped: image-appearance statistics unavailable (run NB 01).")

# Does it reason LESS on severe cases, or just refuse outright?
substantive = abstention[~abstention["abstained"]].dropna(subset=["severity_band"])
if len(substantive):
    print()
    print("Reasoning length among NON-abstained traces (is thinking truncated on severe cases?):")
    print(substantive.groupby("severity_band")["reasoning_chars"]
          .agg(["mean", "median", "count"]).round(0).to_string())

## 6. Answer-format scoring — a legitimate AUROC for A4

NB 07 scored this arm by teacher-forcing `{"covid_positive":"Yes"}`, a string it never emits, so
its `covid_score` was meaningless and correctly quarantined. Scoring in the model's **own**
grammar is a fair contrast and gives A4 a usable continuous score.

The mapping from "lung opacity present" to "PCR-positive" is an author-defined proxy and is
labelled as such: it measures radiographic abnormality, not virology.

In [ ]:
from PIL import Image
from transformers import AutoProcessor

Image.MAX_IMAGE_PIXELS = None
SCORE_PATH = NB11_DIR / "answer_format_scores.jsonl"

# The scoring configuration is part of the cache identity: change a candidate string and every
# cached score becomes meaningless while still looking valid. Same failure mode as NB 08's
# phrase-insensitive entity cache.
SCORE_FINGERPRINT = hashlib.sha256(json.dumps({
    "positive": POSITIVE_ANSWER, "negative": NEGATIVE_ANSWER,
    "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
}, sort_keys=True).encode()).hexdigest()[:16]
FINGERPRINT_PATH = NB11_DIR / "answer_format_scores.fingerprint"

if SCORE_PATH.is_file() and FINGERPRINT_PATH.is_file():
    if FINGERPRINT_PATH.read_text(encoding="utf-8").strip() != SCORE_FINGERPRINT:
        stale = SCORE_PATH.with_suffix(".jsonl.stale")
        SCORE_PATH.replace(stale)
        print(f"Scoring configuration changed; moved the stale cache to {stale.name} and will "
              "rescore. Reusing it would have compared candidates the model was never scored on.")

# NOTE: load_jsonl_by_key returns TUPLE keys, so membership must be tested with (key,).
# Testing a bare string here silently made every run a full rescore while printing a
# reassuring "resuming with N cached" line.
answer_scores = cm.load_jsonl_by_key(SCORE_PATH, ["image_key"]) if SCORE_PATH.is_file() else {}
FINGERPRINT_PATH.write_text(SCORE_FINGERPRINT, encoding="utf-8")
print(f"Resuming with {len(answer_scores):,} cached answer-format scores "
      f"(fingerprint {SCORE_FINGERPRINT}).")

if RUN_ANSWER_FORMAT_SCORING:
    templates = json.loads((NB02_DIR / "prompt_templates.json").read_text())["templates"]
    system_prompt = templates["covid_classification"]["system"]
    user_prompt = templates["covid_classification"]["user"]

    targets = [row for row in internal_cohort.to_dict("records")
               if (str(row["image_key"]),) not in answer_scores]
    if SCORING_MAX_IMAGES is not None:
        targets = targets[:SCORING_MAX_IMAGES]

    if targets:
        from transformers import AutoModelForImageTextToText
        kwargs = {"trust_remote_code": True}
        if MODEL_REVISION:
            kwargs["revision"] = MODEL_REVISION
        processor = AutoProcessor.from_pretrained(MODEL_ID, **kwargs)
        if processor.tokenizer.pad_token_id is None:
            processor.tokenizer.pad_token = processor.tokenizer.eos_token
        # `torch_dtype` was renamed to `dtype`; pick whichever the installed transformers
        # accepts rather than pinning to one and emitting a deprecation warning.
        import inspect
        from transformers import PreTrainedModel
        try:
            dtype_key = ("dtype" if "dtype" in
                         inspect.signature(PreTrainedModel.from_pretrained).parameters
                         else "torch_dtype")
        except Exception:
            dtype_key = "torch_dtype"
        model = AutoModelForImageTextToText.from_pretrained(
            MODEL_ID, **{dtype_key: torch.bfloat16}, device_map="auto",
            low_cpu_mem_usage=True, **kwargs).eval()

        def render(image_path):
            messages = [
                {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
                {"role": "user", "content": [{"type": "image", "image": str(image_path)},
                                             {"type": "text", "text": user_prompt}]}]
            return processor.apply_chat_template(
                messages, add_generation_prompt=True, tokenize=False)

        @torch.inference_mode()
        def score_candidate(image_path, candidate):
            prompt = render(image_path)
            with Image.open(image_path) as handle:
                image = handle.convert("RGB")
                full = processor(text=prompt + candidate, images=image, return_tensors="pt")
                only = processor(text=prompt, images=image, return_tensors="pt")
            device = next(p.device for p in model.parameters() if p.device.type != "meta")
            full = {k: (v.to(device=device, dtype=torch.bfloat16)
                        if v.is_floating_point() else v.to(device)) for k, v in full.items()}
            start = only["input_ids"].shape[-1]
            end = full["input_ids"].shape[-1]
            if end <= start:
                return float("nan")
            logits = model(**full).logits.float()
            log_probs = torch.log_softmax(logits[0, start - 1:end - 1], dim=-1)
            tokens = full["input_ids"][0, start:end]
            return float(log_probs.gather(-1, tokens[:, None]).squeeze(-1).sum())

        started = time.perf_counter()
        for position, row in enumerate(targets, start=1):
            key = str(row["image_key"])
            try:
                positive = score_candidate(row["v0_image"], POSITIVE_ANSWER)
                negative = score_candidate(row["v0_image"], NEGATIVE_ANSWER)
            except Exception as exc:
                print(f"    {key}: {type(exc).__name__}: {exc}")
                continue
            if math.isnan(positive) or math.isnan(negative):
                continue
            maximum = max(positive, negative)
            probability = (math.exp(positive - maximum)
                           / (math.exp(positive - maximum) + math.exp(negative - maximum)))
            record = {"image_key": key, "logL_positive": positive, "logL_negative": negative,
                      "abnormality_score": probability,
                      "gt_covid": row.get("covid_positive"),
                      "gt_mrale_total": row.get("mrale_total_annotated")}
            cm.append_jsonl(SCORE_PATH, record)
            answer_scores[(key,)] = record
            if position % 100 == 0:
                rate = position / max(time.perf_counter() - started, 1e-6)
                print(f"    {position}/{len(targets)} ({rate:.2f} img/s)")

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    else:
        print("All images already scored.")
else:
    print("RUN_ANSWER_FORMAT_SCORING is False; A4 will have no usable continuous score.")

print(f"Answer-format scores: {len(answer_scores):,}")

## 7. Metrics and the restated RQ2

In [ ]:
score_rows = list(answer_scores.values())
predictions = []
for record in score_rows:
    key = record["image_key"]
    source = by_key.get(key, {})
    parsed = parsed_by_key.get(key, {})
    probability = record.get("abnormality_score")
    predictions.append(cm.make_prediction_row(
        image_key=key, cohort="MIDRC", subcohort="MIDRC",
        filename=key.split("::")[-1],
        held_out_fold=source.get("fold"), agent=AGENT_NAME, arm=ARM,
        view="v0", task="joint",
        covid_pred=(None if probability is None else
                    ("Yes" if probability >= 0.5 else "No")),
        covid_score=probability,
        # mRALE is deliberately absent: it is not in this model's output space.
        mrale_total=None,
        valid=not parsed.get("abstained", False),
        parse_error=("abstained" if parsed.get("abstained") else None),
        model_id=MODEL_ID, model_revision=MODEL_REVISION,
        n_findings=parsed.get("n_labels"), abstained=parsed.get("abstained"),
        **ground_truth_fields(source) if source else {},
    ))
cm.write_jsonl(NB11_DIR / "predictions_nvreason.jsonl", predictions)

metrics = evaluate_arm(predictions, ARM)
covid = metrics.get("covid", {})
print("A4 with answer-format scoring:")
print(f"  n              : {metrics['n_rows']:,}")
print(f"  AUROC          : {covid.get('auroc', float('nan')):.4f}")
print(f"  AUPRC          : {covid.get('auprc', float('nan')):.4f}")
print(f"  balanced acc   : {covid.get('balanced_accuracy', float('nan')):.4f}")
print(f"  specificity    : {covid.get('specificity', float('nan')):.4f}")
print()
print("Compare against NB 07's JSON-format score for the same arm (AUROC 0.4951, quarantined).")
print("Any improvement is a measure of how badly a format mismatch corrupts a likelihood score,")
print("which is itself worth one sentence in the protocol 7.2 discussion.")

summary_row = OrderedDict([
    ("arm", ARM), ("agent", AGENT_NAME), ("model_id", MODEL_ID),
    ("adaptation", "zero-shot, scored in its own answer format"),
    ("n_images", metrics["n_rows"]),
    ("mrale_mae_pooled", None), ("mrale_mae_ci95", None),
    ("covid_auroc_pooled", round(covid.get("auroc", float("nan")), 4)),
    ("covid_auprc", round(covid.get("auprc", float("nan")), 4)),
    ("covid_balanced_accuracy", round(covid.get("balanced_accuracy", float("nan")), 4)),
    ("covid_specificity", round(covid.get("specificity", float("nan")), 4)),
    ("covid_sensitivity", round(covid.get("sensitivity", float("nan")), 4)),
    ("abstention_rate", round(float(abstention["abstained"].mean()), 4)),
    ("abstention_severe_band", float(band_table.loc["severe", "rate"])
     if "severe" in band_table.index else None),
    ("mrale_usable", False),
    ("score_usable", None),
    ("report_in_table2", "COVID row only; OMIT mRALE (output space cannot express it)"),
])
pd.DataFrame([summary_row]).to_csv(NB11_DIR / "arm_summary.csv", index=False)

cm.write_json(NB11_DIR / "rq2_restatement.json", {
    "original": "Does reasoning-specialized pretraining improve mRALE assessment?",
    "why_unanswerable": (
        "NV-Reason-CXR-3B emits <think>...</think><answer>finding labels</answer>. mRALE "
        "components are not in its output vocabulary, so the original RQ2 asks the model a "
        "question it cannot express an answer to."
    ),
    "restated": (
        "Does a reasoning-tuned CXR model contribute usable diagnostic evidence, and is its "
        "abstention behaviour a surface artefact of output formatting or a deeper limitation?"
    ),
    "evidence": {
        "abstention_overall": round(float(abstention["abstained"].mean()), 4),
        "abstention_by_band": {b: float(band_table.loc[b, "rate"]) for b in band_table.index},
        "spearman_abstention_vs_mrale": round(float(rho), 4),
        "confound_tested": ["mean_intensity", "stddev_intensity", "min_side"],
        "answer_format_auroc": round(covid.get("auroc", float("nan")), 4),
        "json_format_auroc_nb07": 0.4951,
    },
    "reporting": [
        "Report the COVID row only; omit mRALE and state that the output space cannot express it.",
        "The abstention gradient deserves a figure, not a footnote: it bears on referee 1.5 "
        "(robustness) and 1.4 (failure modes).",
        "A model that abstains hardest where severity assessment matters most is a limitation "
        "worth reporting even though -- especially though -- it is a negative result.",
    ],
})
print()
print("rq2_restatement.json written. The plan's Section 2 needs amending to match.")

## 8. E5-S — reserved, not implemented in this notebook

E5-S remains scientifically useful for asking whether supervision overrides abstention, but
the training implementation is not present here. `RUN_E5S_LORA=True` therefore raises a clear
error and never writes a completion marker. Implement E5-S as a dedicated Qwen2.5-VL training
notebook before enabling it.

The protocol gated E5-S on E0c being competitive, and it was not. But that gate was written before
the abstention finding, and fine-tuning is the one intervention that can change both an output
format and a refusal policy. If LoRA teaches the mRALE schema **and** abstention collapses, the
behaviour was superficial. If the schema is learned but severe cases are still refused or scored
badly, the limitation is deeper — and that is the more interesting result.

One fold should be sufficient for this mechanism question. Any future implementation must save
real adapter weights and must distinguish completion from a recipe or scaffold file.

In [ ]:
if not RUN_E5S_LORA:
    print("E5-S skipped (RUN_E5S_LORA = False).")
    print()
    print("The optional intervention is not implemented in this notebook.")
    print("If you skip it, the manuscript should say that the abstention was characterised but")
    print("not intervened on -- which is honest and still answers the restated RQ2.")
else:
    raise NotImplementedError(
        "E5-S LoRA training is not implemented in NB 11. Leave RUN_E5S_LORA=False, "
        "or implement a dedicated Qwen2.5-VL training notebook that saves and verifies "
        "real adapter weights. No scaffold completion marker has been written."
    )

## 9. Gate

In [ ]:
failures, warnings = [], []

if not traces:
    failures.append("No traces loaded; nothing to analyse.")
if not findings_rows:
    failures.append("No findings parsed from the traces.")

n_answer = sum(1 for row in findings_rows if row["n_findings"])
if n_answer == 0:
    failures.append(
        "No trace contained an <answer> tag, so the assumed output grammar is wrong. Re-read a "
        "few raw traces before trusting anything in this notebook."
    )

if abstention["abstained"].mean() > 0.9:
    warnings.append(
        f"Abstention rate {abstention['abstained'].mean():.1%} is near total. The traces in "
        "NB 07 were generated with a reasoning-style prompt; a schema-only prompt refuses far "
        "more often. Check which prompt produced this file before interpreting the gradient."
    )

if len(band_table) >= 2:
    ordered = [band_table.loc[b, "rate"] for b in
               ["none", "mild", "moderate", "severe"] if b in band_table.index]
    monotone = all(a <= b + 1e-9 for a, b in zip(ordered, ordered[1:]))
    warnings.append(
        f"ABSTENTION GRADIENT: {[round(float(x), 3) for x in ordered]} across severity bands "
        f"({'monotone increasing' if monotone else 'not monotone'}), "
        f"Spearman {rho:+.4f}. This is A4's headline result and belongs in a figure."
    )

usable_score = len(answer_scores) > 0 and not math.isnan(covid.get("auroc", float("nan")))
if RUN_ANSWER_FORMAT_SCORING and not usable_score:
    warnings.append("Answer-format scoring produced no usable AUROC; A4 stays score-unusable.")

cm.write_json(NB11_DIR / "usability.json", {ARM: {
    "score_usable": bool(usable_score),
    "mrale_usable": False,
    "mrale_unusable_reason": "output space cannot express mRALE (see rq2_restatement.json)",
    "abstention_rate": round(float(abstention["abstained"].mean()), 4),
}})

warnings.append(
    "TABLE 2: report the COVID row for this arm and OMIT mRALE entirely. Printing 24.0 would "
    "misreport an output-space incompatibility as a severity result."
)


def report(title, messages):
    print(title)
    for message in messages or []:
        print("  -", message)
    if not messages:
        print("  none")


report("WARNINGS", warnings)
print()
report("FAILURES", failures)

cm.write_json(NB11_DIR / "gate_nb11.json",
              {"passed": not failures, "failures": failures, "warnings": warnings})
if failures:
    detail = "\n".join(f"  [{i + 1}] {m}" for i, m in enumerate(failures))
    raise AssertionError(f"NB 11 gate failed with {len(failures)} blocking issue(s):\n{detail}")
print()
print("NB 11 gate: PASSED")

## Notes carried forward

- **`nvreason_findings.jsonl` is A4's contribution to the roster**, not a severity estimate. NB 15's
  reasoner consumes it the same way it consumes A6's entity findings, and NB 20's grounding audit
  checks reasoning traces against both.
- **The plan's Section 2 needs amending.** `rq2_restatement.json` contains the replacement wording
  and the supporting evidence.
- **E5-S is not implemented in NB 11.** Enabling it raises and writes no completion marker.
  A future implementation should be a dedicated Qwen2.5-VL notebook with real adapter-weight
  verification. Its mechanism question is whether severity-dependent abstention survives
  supervision; one fold should be sufficient.